# Jargon Filtering Pipeline

Constructs the medical jargon dataset (J) and non-jargon dataset (N) used in
the JU/JI benchmarks, as described in **Section 3.1** and **Appendix C–D** of the paper.

**Pipeline:**
1. Load README-exp_good from HuggingFace (`bio-nlp-umass/NoteAid-README`)
2. Clean and normalize jargon terms and definitions
3. Deduplicate near-identical definitions per term using SBERT cosine similarity
4. Load Wiktionary (wiktextract) English entries
5. For each term, compare its README (medical) definition against its
   non-medical Wiktionary senses using SBERT cosine similarity
6. Terms whose medical definition is sufficiently distant from all
   non-medical senses (max similarity < threshold) are kept as jargon;
   the rest are reassigned to the non-jargon set


## Setup

In [ ]:
import ast
import gzip
import json
import re
from pathlib import Path
import os
import pandas as pd
import numpy as np
import torch
from huggingface_hub import snapshot_download
from sentence_transformers import SentenceTransformer, util
from tqdm import trange

In [ ]:
# --- Config ---

SBERT_MODEL_NAME = "all-mpnet-base-v2"
DEDUP_SIM_THRESHOLD = 0.92   # Appendix C: within-term semantic dedup threshold
JARGON_SIM_THRESHOLD = 0.5   # Appendix D: tau, jargon vs. non-medical sense threshold

# Appendix D: Wiktionary tags/topics indicating specialized medical usage.
# Definitions carrying any of these are excluded from the "non-medical sense"
# comparison set, since they are themselves medical/technical senses.
MEDICAL_TAGS = {"jargon", "slang"}
MEDICAL_TOPICS = {
    "medicine", "pathology", "pharmacology", "anatomy",
    "surgery", "immunology", "physiology",
    "biology", "microbiology", "biochemistry",
    "genetics", "zoology", "botany", "entomology",
    "psychology",
}

# --- Paths (edit these for your environment) ---
README_CACHE_DIR = f"/scratch/{os.environ['USER']}/NoteAid_README" #Path("./cache/NoteAid_README")
WIKTEXTRACT_PATH = Path("/share/data/sun-group/dkeng/raw-wiktextract-data.jsonl.gz")  
OUTPUT_DIR = Path("./output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
sbert_model = SentenceTransformer(SBERT_MODEL_NAME)

## Step 1: Load README-exp_good

Downloads the dataset from HuggingFace and selects the expert-annotated,
quality-checked subset (`README_exp_good`).

In [ ]:
def download_readme_dataset(cache_dir: Path) -> Path:
    """Download README-exp_good from HuggingFace and return its local path."""
    local_dir = snapshot_download(
        repo_id="bio-nlp-umass/NoteAid-README",
        repo_type="dataset",
        local_dir=str(cache_dir),
    )
    return Path(local_dir)


def load_readme_exp_good(readme_dir: Path) -> pd.DataFrame:
    """Load and rename columns for the README-exp_good subset (Appendix C)."""
    csv_path = readme_dir / "readme_exp_good.csv"
    df = pd.read_csv(csv_path)
    df = df.rename(columns={
        "ann_text": "jargon_term",
        "split_print": "gen_def",
        "gpt_generated": "lay_def",
        "gpt_text_to_annotate": "EHR",
    })
    df = df[["jargon_term", "gen_def", "EHR"]]
    df = df.drop_duplicates(subset=["jargon_term", "gen_def"])
    return df

In [ ]:
readme_dir = download_readme_dataset(README_CACHE_DIR)
raw_df = load_readme_exp_good(readme_dir)
print(f"Loaded {len(raw_df)} (term, definition) rows")
raw_df.head()

## Step 2: Clean and normalize terms / definitions

Lowercases and strips punctuation from jargon terms, and parses the
stringified definition lists (e.g. `'["def1", "def2"]'`) into real Python lists.

In [ ]:
def clean_term(term: str) -> str:
    """Normalize a jargon term: lowercase, strip whitespace/punctuation,
    fix spaced possessives and dashes."""
    if not isinstance(term, str):
        return ""

    s = term.lower().strip()
    s = re.sub(r"\s+'s\b", "'s", s)          # "hodgkin 's" -> "hodgkin's"
    s = re.sub(r"\s*-\s*", "-", s)           # "t - cell" -> "t-cell"
    s = re.sub(r"^[\s,;:\-\(\[\{'\"`]+", "", s)   # strip leading punctuation
    s = re.sub(r"[\s,;:\)\]\}'\"`]+$", "", s)      # strip trailing punctuation
    s = re.sub(r"\s+", " ", s)               # collapse repeated whitespace
    return s.strip()


def parse_gen_def_list(raw: str) -> list[str]:
    """Parse the stringified list of definitions, e.g. '["def1", "def2"]',
    into an actual list of strings. Falls back to treating the raw string
    as a single definition if parsing fails."""
    if not isinstance(raw, str):
        return []
    try:
        parsed = ast.literal_eval(raw)
        if isinstance(parsed, list):
            return [str(d).strip() for d in parsed if str(d).strip()]
        return [str(parsed).strip()]
    except (ValueError, SyntaxError):
        return [raw.strip()] if raw.strip() else []


def clean_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """Apply term/definition cleaning and drop exact duplicate (term, def) pairs."""
    df = df.copy()
    df["jargon_clean"] = df["jargon_term"].apply(clean_term)
    df["gen_def_clean"] = df["gen_def"].str.replace(
        r'^\s*\[\s*["\']|["\']\s*\]\s*$', "", regex=True
    )
    df["gen_def_list"] = df["gen_def"].apply(parse_gen_def_list)

    before = len(df)
    df = df.drop_duplicates(subset=["jargon_clean", "gen_def_clean"]).reset_index(drop=True)
    print(f"Exact dedup: {before} -> {len(df)} unique (term, def) pairs")
    return df

In [ ]:
clean_df = clean_dataframe(raw_df)
clean_df.head()

## Step 3: Within-term semantic deduplication

Many terms have multiple near-duplicate definitions that differ only in
word order. For each term, greedily keep a definition only if it's not
too similar (SBERT cosine sim < 0.92) to an already-kept definition.

In [ ]:
def semantic_dedup_by_term(df: pd.DataFrame, sbert_model: SentenceTransformer) -> pd.DataFrame:
    """For each jargon term with multiple definitions, greedily keep only
    definitions that are not near-duplicates (cosine sim < DEDUP_SIM_THRESHOLD)
    of an already-kept definition for that term."""
    kept_groups = []

    for _, group in df.groupby("jargon_clean"):
        defs = group["gen_def_clean"].tolist()

        if len(defs) == 1:
            kept_groups.append(group.iloc[[0]])
            continue

        embeddings = sbert_model.encode(defs, convert_to_tensor=True, normalize_embeddings=True)
        sim_matrix = util.cos_sim(embeddings, embeddings)

        kept_indices = [0]
        for i in range(1, len(defs)):
            if all(sim_matrix[i][j].item() < DEDUP_SIM_THRESHOLD for j in kept_indices):
                kept_indices.append(i)

        kept_groups.append(group.iloc[kept_indices])

    deduped = pd.concat(kept_groups, ignore_index=True)
    print(f"Semantic dedup: {len(df)} -> {len(deduped)} rows")
    return deduped

In [ ]:
deduped_df = semantic_dedup_by_term(clean_df, sbert_model)
deduped_df.head()

## Step 4: Load Wiktionary (wiktextract)

Streams the raw wiktextract `.jsonl.gz` dump, keeps English entries, and
indexes them by word, merging all senses per word.

In [ ]:
class Wiki_Word:
    
    def __init__(self, word):
        self.word = word
        self.pos_tags = set()
        self.definitions = []
        
    def attach_def(self, word_def, pos, sentences, tags, topics):
        
        new_def = {'def':word_def, 'pos':pos, 'sents':sentences, 'tags':tags, 'topics':topics}
        self.pos_tags.add(pos)
        self.definitions.append(new_def)

def filter_wiktionary_to_terms(wiki_entries: list[WiktionaryEntry], terms: set[str]) -> dict[str, WiktionaryEntry]:
    """Restrict the Wiktionary index to words appearing in our term set, and
    merge definitions for words that appear under multiple casings/entries."""
    wiki_by_word: dict[str, WiktionaryEntry] = {}

    for entry in wiki_entries:
        key = entry.word.strip().lower()
        if key not in terms:
            continue
        if key in wiki_by_word:
            wiki_by_word[key].definitions.extend(entry.definitions)
        else:
            wiki_by_word[key] = entry

    print(f"Wiktionary terms matched: {len(wiki_by_word)} / {len(terms)}")
    return wiki_by_word

In [ ]:
WIKT_PATH = "path/to/wikt-en.npy"
wiki_data = np.load(WIKT_PATH, allow_pickle=True)

In [ ]:
jargon_term_set = set(deduped_df["jargon_clean"].str.strip().str.lower())
wiki_by_word = filter_wiktionary_to_terms(wiki_entries, jargon_term_set)

## Step 5: Score and label jargon vs. non-jargon

For each term, compute the max cosine similarity between its README
definition and its **non-medical** Wiktionary senses (Appendix D, Eq. 3).
Terms whose definition is sufficiently distant from all non-medical senses
(similarity < threshold) are labeled jargon.

In [ ]:
def is_non_medical_sense(definition: dict) -> bool:
    """Returns True if a Wiktionary definition does NOT carry a medical/jargon
    tag or topic, i.e. it represents the term's conventional, everyday sense."""
    tags = definition.get("tags") or []
    topics = definition.get("topics") or []

    for t in tags:
        if t.lower() in bad_tags:
            return False
    for tp in topics:
        if tp.lower() in bad_topics:
            return False
    return True


def build_non_medical_sense_cache(
    wiki_by_word: dict[str, WiktionaryEntry],
    sbert_model: SentenceTransformer,
) -> dict[str, tuple[list[str], torch.Tensor]]:
    """Pre-compute SBERT embeddings for each term's non-medical Wiktionary senses."""
    cache = {}
    for term, entry in wiki_by_word.items():
        non_medical_defs = [
            d["def"] for d in entry.definitions
            if d.get("def") and is_non_medical_sense(d)
        ]
        if non_medical_defs:
            embeddings = sbert_model.encode(non_medical_defs, convert_to_tensor=True, normalize_embeddings=True)
            cache[term] = (non_medical_defs, embeddings)
    return cache


def score_jargon_candidates(
    df: pd.DataFrame,
    non_medical_cache: dict[str, tuple[list[str], torch.Tensor]],
    wiki_by_word: dict[str, WiktionaryEntry],
    sbert_model: SentenceTransformer,
) -> pd.DataFrame:
    """For each term, compute the max cosine similarity between its README
    definition and its non-medical Wiktionary senses. Terms with no
    Wiktionary entry, or no non-medical sense, are marked accordingly and
    treated as having no similarity score (i.e. default to jargon)."""
    rows = []

    for _, row in df.iterrows():
        term = row["jargon_clean"]
        orig_term = row["jargon_term"]
        gen_defs = row["gen_def_list"]

        if not gen_defs:
            rows.append((term, orig_term, None, "empty_gen_def", None, None))
            continue

        if term not in non_medical_cache:
            status = "no_wiki" if term not in wiki_by_word else "no_non_jargon_gloss"
            rows.append((term, orig_term, None, status, None, gen_defs[0]))
            continue

        non_medical_defs, non_medical_embs = non_medical_cache[term]
        gen_embs = sbert_model.encode(gen_defs, convert_to_tensor=True, normalize_embeddings=True)

        sim = util.cos_sim(gen_embs, non_medical_embs)
        max_sim = torch.max(sim).item()
        max_idx = torch.argmax(sim)
        gen_idx = (max_idx // sim.size(1)).item()
        wiki_idx = (max_idx % sim.size(1)).item()

        rows.append((term, orig_term, max_sim, "ok", non_medical_defs[wiki_idx], gen_defs[gen_idx]))

    return pd.DataFrame(rows, columns=["term", "jargon_term", "max_sim", "status", "wiki_def", "gen_def"])

In [ ]:
non_medical_cache = build_non_medical_sense_cache(wiki_by_word, sbert_model)
scored_df = score_jargon_candidates(deduped_df, non_medical_cache, wiki_by_word, sbert_model)

scored_df["EHR"] = deduped_df["EHR"].values
scored_df["gen_def_clean"] = deduped_df["gen_def_clean"].values
scored_df["jargon_clean"] = deduped_df["jargon_clean"].values

scored_df.head()

In [ ]:
scored_df

In [ ]:
THRESH = 0.5
#res_df = pd.DataFrame(results, columns=["term", "max_sim", "status", "wiki_def"])
scored_df["pred_is_jargon"] = ~(scored_df["max_sim"].fillna(0) >= THRESH)
scored_df

In [ ]:
jargon=scored_df[scored_df['pred_is_jargon']==True]
jargon

In [ ]:
def label_jargon(scored_df: pd.DataFrame, threshold: float = JARGON_SIM_THRESHOLD) -> pd.DataFrame:
    """Apply the jargon/non-jargon threshold (Appendix D, Equation 3).

    A term is jargon if its README definition is sufficiently distant
    (similarity < threshold) from all of its non-medical Wiktionary senses.
    Terms with no similarity score (no Wiktionary match) default to jargon.
    """
    scored_df = scored_df.copy()
    scored_df["pred_is_jargon"] = ~(scored_df["max_sim"].fillna(0) >= threshold)
    return scored_df

In [ ]:
labeled_df = label_jargon(scored_df)

jargon_df = labeled_df[labeled_df["pred_is_jargon"]]
non_jargon_df = labeled_df[~labeled_df["pred_is_jargon"]]

print(f"Final split: {len(jargon_df)} jargon terms, {len(non_jargon_df)} non-jargon terms")

## Save outputs

In [ ]:
labeled_df.to_csv(OUTPUT_DIR / "readme_filtered_all.csv", index=False)
jargon_df.to_csv(OUTPUT_DIR / "readme_filtered_jargon.csv", index=False)
non_jargon_df.to_csv(OUTPUT_DIR / "readme_filtered_non_jargon.csv", index=False)
print(f"Saved outputs to {OUTPUT_DIR}/")